# ResNeXt [2 points]
Implement ResNeXT. It is expected that your accuracy is higher than ResNet. Compare the results with your VGG and ResNet implementation.

## Implement the ResNeXT architecture

In [ ]:
import os, sys, random, time, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
import warnings
warnings.filterwarnings('ignore')

subprocess.run(['pip', 'install', '-q', 'kaggle'], check=False)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = min(os.cpu_count() or 1, 4)
print(f'Device: {device}')

IN_COLAB = 'google.colab' in str(sys.modules)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/cse676_a2/tomato_dataset')
    CKPT_DIR  = Path('/content/drive/MyDrive/cse676_a2/checkpoints')
else:
    DATA_ROOT = Path('./tomato_dataset')
    CKPT_DIR  = Path('./checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle auth (same key as Part 1)
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_027ba87ea784a719aa4bf33c7ce63446'

# Download dataset if needed
import zipfile
DATA_ROOT.mkdir(parents=True, exist_ok=True)
if not any(DATA_ROOT.rglob('*.jpg')):
    r = subprocess.run(['kaggle', 'datasets', 'download',
                        '-d', 'syedhashirali260/tomato-leaf-disease-dataset-6-classes',
                        '-p', str(DATA_ROOT)], capture_output=True, text=True)
    for zf in DATA_ROOT.glob('*.zip'):
        with zipfile.ZipFile(zf) as z: z.extractall(DATA_ROOT)
        zf.unlink()

raw_dataset = torchvision.datasets.ImageFolder(root=str(next(
    d for d in sorted(DATA_ROOT.rglob('*')) if d.is_dir() and
    sum(1 for sd in d.iterdir() if sd.is_dir()) >= 2 and
    any(list(sd.glob('*.jpg'))[:1] for sd in d.iterdir() if sd.is_dir())
)))
CLASSES     = raw_dataset.classes
NUM_CLASSES = len(CLASSES)
print(f'Classes ({NUM_CLASSES}): {CLASSES}  |  Total: {len(raw_dataset)}')

# Transforms
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)
])
val_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class IndexedDataset(Dataset):
    def __init__(self, ds, indices, tf):
        self.ds = ds; self.indices = indices; self.tf = tf
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        path, label = self.ds.samples[self.indices[i]]
        return self.tf(Image.open(path).convert('RGB')), label

all_idx = list(range(len(raw_dataset)))
tv_idx, te_idx = train_test_split(all_idx, test_size=0.2, stratify=raw_dataset.targets, random_state=SEED)
tr_idx, vl_idx = train_test_split(tv_idx,  test_size=0.2, stratify=[raw_dataset.targets[i] for i in tv_idx], random_state=SEED)

def make_loaders(bs=32):
    kw = dict(batch_size=bs, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'), persistent_workers=(NUM_WORKERS>0))
    return (DataLoader(IndexedDataset(raw_dataset, tr_idx, train_tf), shuffle=True,  **kw),
            DataLoader(IndexedDataset(raw_dataset, vl_idx, val_tf),   shuffle=False, **kw),
            DataLoader(IndexedDataset(raw_dataset, te_idx, val_tf),   shuffle=False, **kw))

train_loader, val_loader, test_loader = make_loaders(32)
print(f'Train: {len(tr_idx)} | Val: {len(vl_idx)} | Test: {len(te_idx)}')

# ── ResNeXt-50_32x4d ─────────────────────────────────────────────────────────
"""
ResNeXt (Xie et al., 2017) extends ResNet by replacing standard convolutions with
grouped convolutions — a parameter called 'cardinality'. ResNeXt-50_32x4d uses
32 groups of width 4 per group at each residual block, increasing representational
power without proportionally increasing parameters.
"""
def build_resnext(num_classes=6, pretrained=True):
    if pretrained:
        model = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.IMAGENET1K_V1)
    else:
        model = models.resnext50_32x4d(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

resnext_model = build_resnext(NUM_CLASSES, pretrained=True)
total_p = sum(p.numel() for p in resnext_model.parameters())
print(f'ResNeXt-50_32x4d — Total params: {total_p:,}')
print(resnext_model)


## Train and evaluate your ResNeXt model
Train and evaluate your ResNeXt model on the same dataset used in Part I.

In [ ]:
import time
try:
    from torch.amp import GradScaler, autocast
except ImportError:
    from torch.cuda.amp import GradScaler, autocast

USE_AMP = device.type == 'cuda'
wandb.login(key="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")
WANDB_PROJECT = 'cse676-a2-bonus'

def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, enabled=USE_AMP):
            out = model(imgs); loss = criterion(out, labels)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct += out.argmax(1).eq(labels).sum().item(); total += labels.size(0)
    return total_loss / total, 100. * correct / total

def eval_epoch(model, loader, criterion):
    model.eval(); total_loss, correct, total = 0., 0, 0; all_p, all_l = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with autocast(device_type=device.type, enabled=USE_AMP):
                out = model(imgs); loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            p = out.argmax(1); correct += p.eq(labels).sum().item(); total += labels.size(0)
            all_p.extend(p.cpu().numpy()); all_l.extend(labels.cpu().numpy())
    return total_loss / total, 100. * correct / total, all_p, all_l

NUM_EPOCHS = 20
criterion  = nn.CrossEntropyLoss()

# Phase 1: train FC only
for p in resnext_model.parameters(): p.requires_grad = False
resnext_model.fc.weight.requires_grad = True; resnext_model.fc.bias.requires_grad = True

opt = optim.Adam(filter(lambda p: p.requires_grad, resnext_model.parameters()), lr=1e-3, weight_decay=1e-4)
sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)
try:
    scaler = GradScaler(device_type=device.type, enabled=USE_AMP)
except TypeError:
    scaler = GradScaler(enabled=USE_AMP)

run = wandb.init(project=WANDB_PROJECT, name='resnext_train', reinit=True)
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_acc, best_state, no_improve = 0., None, 0
PATIENCE = 7

print('=== Phase 1: FC head only (5 epochs) ===')
for ep in range(1, 6):
    t0 = time.time()
    tl, ta = train_epoch(resnext_model, train_loader, criterion, opt, scaler)
    vl, va, _, _ = eval_epoch(resnext_model, val_loader, criterion)
    sch.step(); elapsed = time.time() - t0
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    wandb.log({'epoch': ep, 'train/loss': tl, 'train/acc': ta, 'val/loss': vl, 'val/acc': va})
    print(f'Ph1 Ep {ep}/5 | Tr {tl:.4f}/{ta:.1f}% | Vl {vl:.4f}/{va:.1f}% | {elapsed:.1f}s')
    if va > best_acc:
        best_acc = va; best_state = {k: v.cpu().clone() for k, v in resnext_model.state_dict().items()}

# Phase 2: full fine-tune
for p in resnext_model.parameters(): p.requires_grad = True
opt2 = optim.Adam(resnext_model.parameters(), lr=1e-4, weight_decay=1e-4)
sch2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=NUM_EPOCHS)

print('\n=== Phase 2: Full fine-tuning ===')
no_improve = 0
for ep in range(6, 6 + NUM_EPOCHS):
    t0 = time.time()
    tl, ta = train_epoch(resnext_model, train_loader, criterion, opt2, scaler)
    vl, va, _, _ = eval_epoch(resnext_model, val_loader, criterion)
    sch2.step(); elapsed = time.time() - t0
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    wandb.log({'epoch': ep, 'train/loss': tl, 'train/acc': ta, 'val/loss': vl, 'val/acc': va})
    print(f'Ph2 Ep {ep-5}/{NUM_EPOCHS} | Tr {tl:.4f}/{ta:.1f}% | Vl {vl:.4f}/{va:.1f}% | {elapsed:.1f}s | no_imp={no_improve}')
    if va > best_acc:
        best_acc = va; best_state = {k: v.cpu().clone() for k, v in resnext_model.state_dict().items()}; no_improve = 0
    else:
        no_improve += 1
    if no_improve >= PATIENCE:
        print(f'Early stopping at epoch {ep-5}'); break

run.finish()
resnext_model.load_state_dict(best_state)
print(f'\nBest Val Acc: {best_acc:.2f}%')

# ── Evaluation ─────────────────────────────────────────────────────────────────
rn_tr_loss, rn_tr_acc, _, _ = eval_epoch(resnext_model, train_loader, criterion)
rn_vl_loss, rn_vl_acc, _, _ = eval_epoch(resnext_model, val_loader,   criterion)
rn_te_loss, rn_te_acc, preds, true = eval_epoch(resnext_model, test_loader, criterion)
print(f'\nResNeXt-50 Final | Train: {rn_tr_acc:.2f}% | Val: {rn_vl_acc:.2f}% | Test: {rn_te_acc:.2f}%')

# Learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_acc'], label='Train', color='purple'); ax1.plot(history['val_acc'], label='Val', color='orange')
ax1.set_title('ResNeXt-50 Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['train_loss'], label='Train', color='purple'); ax2.plot(history['val_loss'], label='Val', color='orange')
ax2.set_title('ResNeXt-50 Loss'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout(); plt.savefig('resnext_curves.svg', format='svg', bbox_inches='tight'); plt.show()

# Confusion matrix
cm = confusion_matrix(true, preds)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('ResNeXt-50 Confusion Matrix')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig('resnext_cm.svg', format='svg', bbox_inches='tight'); plt.show()

print('\nClassification Report (ResNeXt-50):')
print(classification_report(true, preds, target_names=CLASSES))
rn_prec, rn_rec, rn_f1, _ = precision_recall_fscore_support(true, preds, average='weighted')
print(f'Weighted — Precision: {rn_prec:.4f}  Recall: {rn_rec:.4f}  F1: {rn_f1:.4f}')

# Log to wandb
run = wandb.init(project=WANDB_PROJECT, name='resnext_eval', reinit=True)
wandb.log({'test/acc': rn_te_acc, 'test/f1': rn_f1})
wandb.log({'confusion_matrix': wandb.plot.confusion_matrix(probs=None, y_true=true, preds=preds, class_names=CLASSES)})
run.finish()

print(f'\nResNeXt-50_32x4d Test Accuracy: {rn_te_acc:.2f}%')
print('ResNeXt uses grouped convolutions (cardinality=32) to increase representational')
print('power over ResNet while keeping parameter count similar. Expected to exceed ResNet-18.')
